# Phase 5–6: Machine Learning & Feature Importance

Compare Logistic Regression, Random Forest, and XGBoost for churn prediction.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import DATA_PROCESSED, IMAGES_DIR, METRICS_PATH, MODEL_PATH, PREDICTIONS_PATH
from src.features import add_engineered_features, get_model_features
from src.ml_models import extract_feature_importance, save_metrics, train_models

In [ ]:
df = add_engineered_features(pd.read_csv(DATA_PROCESSED))
best_model, metrics_df, predictions = train_models(df)
metrics_df

## Model selection

The best model is selected primarily by **ROC-AUC**, with **recall** as a secondary business criterion because missing a churner is costlier than a false alarm.

In [ ]:
importance = extract_feature_importance(best_model, df[get_model_features()])
importance.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=importance.head(12), y="feature", x="importance", palette="viridis", ax=ax)
ax.set_title("Top Churn Drivers")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "feature_importance.png", dpi=150)
plt.show()

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
save_metrics(metrics_df, METRICS_PATH)
predictions.to_csv(PREDICTIONS_PATH, index=False)
print(f"Saved model to {MODEL_PATH}")

## Business interpretation

- Contract type and tenure remain the strongest churn drivers.
- Monthly charges and payment method influence churn risk.
- Month-to-month contracts exhibit significantly higher churn risk than annual contracts.